# Notebook 28 — DINOv2-S Student Nuisance Accessibility Audit

## Objective

Notebook 27 established a strong and stable cross-script writer-verification baseline using a frozen DINOv2-S representation and a small trainable projection head.

The three-seed development result reached approximately **0.8375 cross-script macro AUC**, and the subsequently frozen 181-writer refit model achieved approximately **0.8538 macro AUC** on the untouched 45-writer monitor set.

The next question is no longer whether DINOv2-S contains useful writer information.

Instead, this notebook asks:

> How accessible are script and page/text-condition cues in the DINOv2-S representation, and does the learned writer projection suppress, preserve, or amplify those nuisance cues?

This is a diagnostic representation audit rather than a new writer-verification training experiment.

## Representations Compared

Two representations are compared under the same writer-disjoint probe protocol.

### 1. Cached DINOv2-S Representation

**image → DINOv2-S → L2-normalized 384-D cached feature**

These are the same cached normalized DINOv2-S features audited in Notebook 26 and used as the input representation in Notebook 27.

### 2. Learned Projection Representation

**cached normalized DINOv2-S 384-D feature → Linear 384→144 → L2 normalization**

For this audit, the projection must come from the development-stage model trained only on the **145 fit writers**.

The final 181-writer refit checkpoint from Notebook 27 is intentionally not used because the 36 selection writers were included in that refit. Using it here would violate the writer-disjoint probe evaluation protocol.

The seed-42 fixed epoch-10 development projection is therefore used as the primary projected representation.

## Probe Protocol

All nuisance probes follow the same writer-disjoint split:

- Probe fitting writers: **145 development-fit writers**
- Probe evaluation writers: **36 development-selection writers**
- Writer overlap between probe fit and evaluation: **0**
- Monitor writers used: **No**
- Validation split used: **No**
- Official test split used: **No**

The probes are deliberately simple linear classifiers.

Their purpose is not to maximize nuisance-classification performance, but to measure how linearly accessible nuisance information is in each representation.

## Nuisance Variables

Two nuisance variables are examined.

### Script

Binary classification:

- Arabic
- English

This measures how strongly script identity can be decoded from the representation.

### Text Condition

Binary classification:

- variable text
- same/fixed text

In QUWI, the page structure is:

- page 1: Arabic variable text
- page 2: Arabic same/fixed text
- page 3: English variable text
- page 4: English same/fixed text

This probe therefore measures whether the representation exposes systematic page/text-condition information beyond writer identity.

## Main Comparisons

For each nuisance variable, the audit compares:

**cached DINOv2-S → projected DINOv2-S student**

The key quantities are writer-disjoint probe ROC-AUC, classification accuracy, and the change in nuisance accessibility after projection.

A reduction after projection would indicate that the writer metric-learning head suppresses some linearly accessible nuisance information.

Little or no reduction would indicate that strong writer verification can coexist with substantial nuisance accessibility.

An increase would indicate that the projection may actually make the nuisance variable more linearly separable.

## Interpretation Boundary

A linear probe measures **accessibility**, not causal dependence.

Therefore:

- high script-probe AUC does not by itself prove that the verifier relies on script,
- low probe AUC does not prove complete script invariance,
- a reduction in probe performance does not prove that all nuisance information has been removed.

The result will be used only to determine whether explicit nuisance suppression is scientifically justified as a future experiment.

No architecture, learning rate, epoch budget, or writer-verification model will be changed based on intermediate probe results within this notebook.

## Decision Goal

If nuisance information remains strongly accessible after the learned projection, a subsequent controlled nuisance-suppression experiment may be justified.

If the projection already substantially suppresses nuisance accessibility while preserving strong writer verification, additional adversarial machinery may be unnecessary.

This notebook therefore serves as a diagnostic bridge between the strong DINOv2-S baseline and any future invariance-oriented experiment.

In [1]:
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [2]:
ROOT = Path.cwd().resolve()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SPLIT_PATH = (
    ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

ROLE_PATH = (
    ROOT
    / "reports"
    / "cross_script_writer_geometry_consistency"
    / "development_internal_writer_roles_seed42.csv"
)

DINO_S_ARCHIVE_PATH = (
    ROOT
    / "reports"
    / "dinov2_efficiency_frontier"
    / "dinov2_vits14_reg_embeddings.npz"
)

DEVELOPMENT_CHECKPOINT_PATH = (
    ROOT
    / "checkpoints"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_seed42_best.pt"
)

SEED42_SUMMARY_PATH = (
    ROOT
    / "reports"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_seed42_summary.json"
)

SEED42_HISTORY_PATH = (
    ROOT
    / "reports"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_seed42_history.csv"
)

FINAL_REFIT_CHECKPOINT_PATH = (
    ROOT
    / "checkpoints"
    / "dinov2s_student_baseline"
    / "dinov2s_projection_final_refit_seed42_epoch10.pt"
)

REPORT_DIR = (
    ROOT
    / "reports"
    / "dinov2s_student_nuisance_accessibility_audit"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

required_paths = {
    "split": SPLIT_PATH,
    "roles": ROLE_PATH,
    "dino_s_archive": DINO_S_ARCHIVE_PATH,
    "development_checkpoint": DEVELOPMENT_CHECKPOINT_PATH,
    "seed42_summary": SEED42_SUMMARY_PATH,
    "seed42_history": SEED42_HISTORY_PATH,
}

missing_paths = {
    name: str(path)
    for name, path in required_paths.items()
    if not path.exists()
}

if len(missing_paths) != 0:
    raise FileNotFoundError(
        json.dumps(
            missing_paths,
            indent=2,
        )
    )

split_df = pd.read_csv(
    SPLIT_PATH
)

role_df = pd.read_csv(
    ROLE_PATH
)

seed42_history_df = pd.read_csv(
    SEED42_HISTORY_PATH
)

with open(
    SEED42_SUMMARY_PATH,
    "r",
) as file:
    seed42_summary = json.load(
        file
    )

development_checkpoint = torch.load(
    DEVELOPMENT_CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

role_column = (
    "internal_role"
)

if role_column not in role_df.columns:
    raise RuntimeError(
        "Expected internal_role column was not found."
    )

fit_role_df = (
    role_df[
        role_df[
            role_column
        ] == "fit"
    ]
    .copy()
)

selection_role_df = (
    role_df[
        role_df[
            role_column
        ] == "selection"
    ]
    .copy()
)

fit_writer_ids = set(
    fit_role_df[
        "writer"
    ]
    .astype(int)
    .unique()
)

selection_writer_ids = set(
    selection_role_df[
        "writer"
    ]
    .astype(int)
    .unique()
)

fit_filename_set = set(
    fit_role_df[
        "filename"
    ]
    .astype(str)
)

selection_filename_set = set(
    selection_role_df[
        "filename"
    ]
    .astype(str)
)

checkpoint_epoch = int(
    development_checkpoint[
        "epoch"
    ]
)

checkpoint_training_seed = int(
    development_checkpoint[
        "training_seed"
    ]
)

checkpoint_fit_writers = int(
    development_checkpoint[
        "fit_writers"
    ]
)

checkpoint_selection_writers = int(
    development_checkpoint[
        "selection_writers"
    ]
)

checkpoint_projection_dimension = int(
    development_checkpoint[
        "projection_dimension"
    ]
)

checkpoint_learning_rate = float(
    development_checkpoint[
        "learning_rate"
    ]
)

checkpoint_weight_decay = float(
    development_checkpoint[
        "weight_decay"
    ]
)

checkpoint_source_normalized = bool(
    development_checkpoint[
        "source_features_l2_normalized"
    ]
)

history_last_epoch = int(
    seed42_history_df[
        "epoch"
    ].max()
)

history_epoch10_row = (
    seed42_history_df[
        seed42_history_df[
            "epoch"
        ] == 10
    ]
)

summary_best_epoch = int(
    seed42_summary[
        "best_epoch"
    ]
)

final_refit_checkpoint_exists = bool(
    FINAL_REFIT_CHECKPOINT_PATH.exists()
)

same_checkpoint_as_final_refit = bool(
    DEVELOPMENT_CHECKPOINT_PATH.resolve()
    == FINAL_REFIT_CHECKPOINT_PATH.resolve()
)

development_checkpoint_audit = {
    "notebook": 28,
    "audit": (
        "DINOv2-S development representation resource audit"
    ),
    "split_path": str(
        SPLIT_PATH.relative_to(
            ROOT
        )
    ),
    "role_path": str(
        ROLE_PATH.relative_to(
            ROOT
        )
    ),
    "dino_s_archive_path": str(
        DINO_S_ARCHIVE_PATH.relative_to(
            ROOT
        )
    ),
    "development_checkpoint_path": str(
        DEVELOPMENT_CHECKPOINT_PATH.relative_to(
            ROOT
        )
    ),
    "final_refit_checkpoint_exists": bool(
        final_refit_checkpoint_exists
    ),
    "development_checkpoint_is_final_refit_checkpoint": bool(
        same_checkpoint_as_final_refit
    ),
    "fit_pages": int(
        len(
            fit_role_df
        )
    ),
    "fit_writers": int(
        len(
            fit_writer_ids
        )
    ),
    "selection_pages": int(
        len(
            selection_role_df
        )
    ),
    "selection_writers": int(
        len(
            selection_writer_ids
        )
    ),
    "fit_selection_writer_overlap": int(
        len(
            fit_writer_ids
            & selection_writer_ids
        )
    ),
    "fit_selection_filename_overlap": int(
        len(
            fit_filename_set
            & selection_filename_set
        )
    ),
    "checkpoint_epoch": int(
        checkpoint_epoch
    ),
    "history_last_epoch": int(
        history_last_epoch
    ),
    "summary_best_epoch": int(
        summary_best_epoch
    ),
    "checkpoint_training_seed": int(
        checkpoint_training_seed
    ),
    "checkpoint_fit_writers": int(
        checkpoint_fit_writers
    ),
    "checkpoint_selection_writers": int(
        checkpoint_selection_writers
    ),
    "checkpoint_projection_dimension": int(
        checkpoint_projection_dimension
    ),
    "checkpoint_learning_rate": float(
        checkpoint_learning_rate
    ),
    "checkpoint_weight_decay": float(
        checkpoint_weight_decay
    ),
    "checkpoint_source_features_l2_normalized": bool(
        checkpoint_source_normalized
    ),
    "checkpoint_monitor_used": bool(
        development_checkpoint[
            "monitor_used"
        ]
    ),
    "checkpoint_validation_used": bool(
        development_checkpoint[
            "validation_used"
        ]
    ),
    "checkpoint_official_test_used": bool(
        development_checkpoint[
            "official_test_used"
        ]
    ),
    "epoch10_history_row_present": bool(
        len(
            history_epoch10_row
        ) == 1
    ),
    "use_checkpoint_as_fixed_epoch10_representation": True,
    "checkpoint_selection_result_not_used_for_probe_tuning": True,
    "final_refit_checkpoint_will_not_be_used": True,
    "monitor_will_not_be_used": True,
    "validation_will_not_be_used": True,
    "official_test_will_not_be_used": True,
    "probe_training_started": False,
    "probe_evaluation_started": False,
}

with open(
    REPORT_DIR
    / "dinov2s_nuisance_resource_checkpoint_audit.json",
    "w",
) as file:
    json.dump(
        development_checkpoint_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        development_checkpoint_audit,
        indent=2,
    )
)

if (
    len(
        fit_role_df
    ) != 580
    or len(
        fit_writer_ids
    ) != 145
    or len(
        selection_role_df
    ) != 144
    or len(
        selection_writer_ids
    ) != 36
    or len(
        fit_writer_ids
        & selection_writer_ids
    ) != 0
    or len(
        fit_filename_set
        & selection_filename_set
    ) != 0
    or checkpoint_epoch != 10
    or history_last_epoch != 10
    or summary_best_epoch != 10
    or checkpoint_training_seed != 42
    or checkpoint_fit_writers != 145
    or checkpoint_selection_writers != 36
    or checkpoint_projection_dimension != 144
    or not checkpoint_source_normalized
    or same_checkpoint_as_final_refit
    or development_checkpoint[
        "monitor_used"
    ]
    or development_checkpoint[
        "validation_used"
    ]
    or development_checkpoint[
        "official_test_used"
    ]
):
    raise RuntimeError(
        "Notebook 28 development checkpoint audit failed."
    )

{
  "notebook": 28,
  "audit": "DINOv2-S development representation resource audit",
  "split_path": "splits/quwi_writer_disjoint_split_seed42.csv",
  "role_path": "reports/cross_script_writer_geometry_consistency/development_internal_writer_roles_seed42.csv",
  "dino_s_archive_path": "reports/dinov2_efficiency_frontier/dinov2_vits14_reg_embeddings.npz",
  "development_checkpoint_path": "checkpoints/dinov2s_student_baseline/dinov2s_projection_seed42_best.pt",
  "final_refit_checkpoint_exists": true,
  "development_checkpoint_is_final_refit_checkpoint": false,
  "fit_pages": 580,
  "fit_writers": 145,
  "selection_pages": 144,
  "selection_writers": 36,
  "fit_selection_writer_overlap": 0,
  "fit_selection_filename_overlap": 0,
  "checkpoint_epoch": 10,
  "history_last_epoch": 10,
  "summary_best_epoch": 10,
  "checkpoint_training_seed": 42,
  "checkpoint_fit_writers": 145,
  "checkpoint_selection_writers": 36,
  "checkpoint_projection_dimension": 144,
  "checkpoint_learning_rate": 0.00

In [3]:
REFIT_SOURCE_FEATURE_PATH = (
    ROOT
    / "reports"
    / "dinov2s_student_baseline"
    / "dinov2s_final_refit_aligned_features.npz"
)

ALIGNED_REPRESENTATION_PATH = (
    REPORT_DIR
    / "dinov2s_nuisance_fit_selection_representations.npz"
)

if not REFIT_SOURCE_FEATURE_PATH.exists():
    raise FileNotFoundError(
        str(
            REFIT_SOURCE_FEATURE_PATH
        )
    )

source_artifact = np.load(
    REFIT_SOURCE_FEATURE_PATH,
    allow_pickle=False,
)

required_source_keys = {
    "refit_embeddings",
    "refit_filenames",
    "refit_writers",
    "refit_page_ids",
}

if not required_source_keys.issubset(
    source_artifact.files
):
    raise RuntimeError(
        "Expected fit-selection source feature keys were not found."
    )

source_embeddings = (
    source_artifact[
        "refit_embeddings"
    ]
    .astype(
        np.float32,
        copy=False,
    )
)

source_filenames = (
    source_artifact[
        "refit_filenames"
    ]
    .astype(str)
)

source_writers = (
    source_artifact[
        "refit_writers"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

source_page_ids = (
    source_artifact[
        "refit_page_ids"
    ]
    .astype(
        np.int64,
        copy=False,
    )
)

source_feature_norms = np.linalg.norm(
    source_embeddings,
    axis=1,
)

if (
    source_embeddings.shape
    != (
        724,
        384,
    )
    or len(
        source_filenames
    ) != 724
    or len(
        np.unique(
            source_filenames
        )
    ) != 724
    or len(
        np.unique(
            source_writers
        )
    ) != 181
    or not np.allclose(
        source_feature_norms,
        1.0,
        rtol=0.0,
        atol=1e-5,
    )
):
    raise RuntimeError(
        "Fit-selection source feature artifact audit failed."
    )

probe_role_df = (
    role_df[
        role_df[
            role_column
        ].isin(
            [
                "fit",
                "selection",
            ]
        )
    ][
        [
            "filename",
            "writer",
            role_column,
        ]
    ]
    .copy()
)

split_probe_metadata = (
    split_df[
        [
            "filename",
            "writer",
            "page_id",
            "language",
            "same_text",
        ]
    ]
    .copy()
)

probe_rows = (
    probe_role_df
    .merge(
        split_probe_metadata,
        on="filename",
        how="inner",
        suffixes=(
            "_role",
            "_split",
        ),
        validate="one_to_one",
    )
)

if not (
    probe_rows[
        "writer_role"
    ]
    .astype(int)
    .to_numpy()
    ==
    probe_rows[
        "writer_split"
    ]
    .astype(int)
    .to_numpy()
).all():
    raise RuntimeError(
        "Writer metadata mismatch between role and split files."
    )

probe_rows[
    "writer"
] = (
    probe_rows[
        "writer_role"
    ]
    .astype(int)
)

probe_rows = (
    probe_rows[
        [
            "filename",
            "writer",
            "page_id",
            "language",
            "same_text",
            role_column,
        ]
    ]
    .copy()
)

probe_rows[
    "filename"
] = (
    probe_rows[
        "filename"
    ]
    .astype(str)
)

probe_rows[
    "page_id"
] = (
    probe_rows[
        "page_id"
    ]
    .astype(int)
)

probe_rows[
    "script_label"
] = np.where(
    probe_rows[
        "page_id"
    ].isin(
        [
            3,
            4,
        ]
    ),
    1,
    0,
).astype(
    np.int64
)

probe_rows[
    "text_condition_label"
] = np.where(
    probe_rows[
        "page_id"
    ].isin(
        [
            2,
            4,
        ]
    ),
    1,
    0,
).astype(
    np.int64
)

probe_rows[
    "script_name"
] = np.where(
    probe_rows[
        "script_label"
    ] == 0,
    "Arabic",
    "English",
)

probe_rows[
    "text_condition_name"
] = np.where(
    probe_rows[
        "text_condition_label"
    ] == 0,
    "variable",
    "same_fixed",
)

language_lower = (
    probe_rows[
        "language"
    ]
    .astype(str)
    .str.lower()
)

language_script_label = np.where(
    language_lower.str.contains(
        "english"
    ),
    1,
    0,
).astype(
    np.int64
)

if not np.array_equal(
    probe_rows[
        "script_label"
    ].to_numpy(),
    language_script_label,
):
    raise RuntimeError(
        "Page-derived script labels do not match language metadata."
    )

source_filename_set = set(
    source_filenames
)

probe_filename_set = set(
    probe_rows[
        "filename"
    ]
)

if source_filename_set != probe_filename_set:
    raise RuntimeError(
        "Fit-selection filenames do not exactly match the source feature artifact."
    )

source_feature_lookup = {
    filename: feature
    for filename, feature in zip(
        source_filenames,
        source_embeddings,
    )
}

fit_rows = (
    probe_rows[
        probe_rows[
            role_column
        ] == "fit"
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

selection_rows = (
    probe_rows[
        probe_rows[
            role_column
        ] == "selection"
    ]
    .sort_values(
        [
            "writer",
            "page_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

fit_raw_features = np.stack(
    [
        source_feature_lookup[
            filename
        ]
        for filename in (
            fit_rows[
                "filename"
            ]
        )
    ],
    axis=0,
).astype(
    np.float32,
    copy=False,
)

selection_raw_features = np.stack(
    [
        source_feature_lookup[
            filename
        ]
        for filename in (
            selection_rows[
                "filename"
            ]
        )
    ],
    axis=0,
).astype(
    np.float32,
    copy=False,
)

fit_script_labels = (
    fit_rows[
        "script_label"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

selection_script_labels = (
    selection_rows[
        "script_label"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

fit_text_condition_labels = (
    fit_rows[
        "text_condition_label"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

selection_text_condition_labels = (
    selection_rows[
        "text_condition_label"
    ]
    .to_numpy(
        dtype=np.int64
    )
)


class DINOv2SProjectionAuditModel(
    nn.Module
):
    def __init__(
        self,
        input_dimension=384,
        projection_dimension=144,
    ):
        super().__init__()

        self.projection = nn.Linear(
            input_dimension,
            projection_dimension,
            bias=True,
        )

    def forward(
        self,
        features,
    ):
        features = F.normalize(
            features,
            p=2,
            dim=-1,
        )

        projected = (
            self.projection(
                features
            )
        )

        projected = F.normalize(
            projected,
            p=2,
            dim=-1,
        )

        return projected


projection_model = (
    DINOv2SProjectionAuditModel(
        input_dimension=384,
        projection_dimension=144,
    )
)

projection_model.load_state_dict(
    development_checkpoint[
        "model_state_dict"
    ],
    strict=True,
)

projection_model.eval()

with torch.no_grad():
    fit_projected_features = (
        projection_model(
            torch.from_numpy(
                fit_raw_features
            ).float()
        )
        .cpu()
        .numpy()
    )

    selection_projected_features = (
        projection_model(
            torch.from_numpy(
                selection_raw_features
            ).float()
        )
        .cpu()
        .numpy()
    )

fit_raw_norms = np.linalg.norm(
    fit_raw_features,
    axis=1,
)

selection_raw_norms = np.linalg.norm(
    selection_raw_features,
    axis=1,
)

fit_projected_norms = np.linalg.norm(
    fit_projected_features,
    axis=1,
)

selection_projected_norms = np.linalg.norm(
    selection_projected_features,
    axis=1,
)

fit_writer_set = set(
    fit_rows[
        "writer"
    ]
    .astype(int)
)

selection_writer_set = set(
    selection_rows[
        "writer"
    ]
    .astype(int)
)

np.savez_compressed(
    ALIGNED_REPRESENTATION_PATH,
    fit_raw_features=(
        fit_raw_features
    ),
    selection_raw_features=(
        selection_raw_features
    ),
    fit_projected_features=(
        fit_projected_features
    ),
    selection_projected_features=(
        selection_projected_features
    ),
    fit_filenames=np.asarray(
        fit_rows[
            "filename"
        ]
        .astype(str)
        .tolist(),
        dtype=str,
    ),
    selection_filenames=np.asarray(
        selection_rows[
            "filename"
        ]
        .astype(str)
        .tolist(),
        dtype=str,
    ),
    fit_writers=np.asarray(
        fit_rows[
            "writer"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
    selection_writers=np.asarray(
        selection_rows[
            "writer"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
    fit_page_ids=np.asarray(
        fit_rows[
            "page_id"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
    selection_page_ids=np.asarray(
        selection_rows[
            "page_id"
        ]
        .astype(int)
        .to_numpy(),
        dtype=np.int64,
    ),
    fit_script_labels=(
        fit_script_labels
    ),
    selection_script_labels=(
        selection_script_labels
    ),
    fit_text_condition_labels=(
        fit_text_condition_labels
    ),
    selection_text_condition_labels=(
        selection_text_condition_labels
    ),
)

representation_alignment_audit = {
    "source_artifact": str(
        REFIT_SOURCE_FEATURE_PATH.relative_to(
            ROOT
        )
    ),
    "source_artifact_pages": int(
        len(
            source_embeddings
        )
    ),
    "source_artifact_writers": int(
        len(
            np.unique(
                source_writers
            )
        )
    ),
    "source_artifact_contains_only_fit_and_selection": True,
    "fit_pages": int(
        len(
            fit_rows
        )
    ),
    "fit_writers": int(
        len(
            fit_writer_set
        )
    ),
    "selection_pages": int(
        len(
            selection_rows
        )
    ),
    "selection_writers": int(
        len(
            selection_writer_set
        )
    ),
    "fit_selection_writer_overlap": int(
        len(
            fit_writer_set
            & selection_writer_set
        )
    ),
    "raw_feature_dimension": int(
        fit_raw_features.shape[
            1
        ]
    ),
    "projected_feature_dimension": int(
        fit_projected_features.shape[
            1
        ]
    ),
    "fit_raw_shape": list(
        fit_raw_features.shape
    ),
    "selection_raw_shape": list(
        selection_raw_features.shape
    ),
    "fit_projected_shape": list(
        fit_projected_features.shape
    ),
    "selection_projected_shape": list(
        selection_projected_features.shape
    ),
    "fit_raw_norm_min": float(
        fit_raw_norms.min()
    ),
    "fit_raw_norm_mean": float(
        fit_raw_norms.mean()
    ),
    "fit_raw_norm_max": float(
        fit_raw_norms.max()
    ),
    "selection_raw_norm_min": float(
        selection_raw_norms.min()
    ),
    "selection_raw_norm_mean": float(
        selection_raw_norms.mean()
    ),
    "selection_raw_norm_max": float(
        selection_raw_norms.max()
    ),
    "fit_projected_norm_min": float(
        fit_projected_norms.min()
    ),
    "fit_projected_norm_mean": float(
        fit_projected_norms.mean()
    ),
    "fit_projected_norm_max": float(
        fit_projected_norms.max()
    ),
    "selection_projected_norm_min": float(
        selection_projected_norms.min()
    ),
    "selection_projected_norm_mean": float(
        selection_projected_norms.mean()
    ),
    "selection_projected_norm_max": float(
        selection_projected_norms.max()
    ),
    "fit_script_counts": {
        "Arabic": int(
            (
                fit_script_labels
                == 0
            ).sum()
        ),
        "English": int(
            (
                fit_script_labels
                == 1
            ).sum()
        ),
    },
    "selection_script_counts": {
        "Arabic": int(
            (
                selection_script_labels
                == 0
            ).sum()
        ),
        "English": int(
            (
                selection_script_labels
                == 1
            ).sum()
        ),
    },
    "fit_text_condition_counts": {
        "variable": int(
            (
                fit_text_condition_labels
                == 0
            ).sum()
        ),
        "same_fixed": int(
            (
                fit_text_condition_labels
                == 1
            ).sum()
        ),
    },
    "selection_text_condition_counts": {
        "variable": int(
            (
                selection_text_condition_labels
                == 0
            ).sum()
        ),
        "same_fixed": int(
            (
                selection_text_condition_labels
                == 1
            ).sum()
        ),
    },
    "script_labels_verified_against_language_metadata": True,
    "text_condition_labels_derived_from_page_protocol": True,
    "development_projection_checkpoint_epoch": int(
        development_checkpoint[
            "epoch"
        ]
    ),
    "development_projection_training_seed": int(
        development_checkpoint[
            "training_seed"
        ]
    ),
    "raw_features_unit_normalized": bool(
        np.allclose(
            fit_raw_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
        and np.allclose(
            selection_raw_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
    ),
    "projected_features_unit_normalized": bool(
        np.allclose(
            fit_projected_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
        and np.allclose(
            selection_projected_norms,
            1.0,
            rtol=0.0,
            atol=1e-5,
        )
    ),
    "aligned_artifact": str(
        ALIGNED_REPRESENTATION_PATH.relative_to(
            ROOT
        )
    ),
    "monitor_features_loaded": False,
    "monitor_evaluated": False,
    "validation_used": False,
    "official_test_used": False,
    "probe_training_started": False,
    "probe_evaluation_started": False,
}

with open(
    REPORT_DIR
    / "dinov2s_nuisance_representation_alignment_audit.json",
    "w",
) as file:
    json.dump(
        representation_alignment_audit,
        file,
        indent=2,
    )

print(
    json.dumps(
        representation_alignment_audit,
        indent=2,
    )
)

if (
    fit_raw_features.shape
    != (
        580,
        384,
    )
    or selection_raw_features.shape
    != (
        144,
        384,
    )
    or fit_projected_features.shape
    != (
        580,
        144,
    )
    or selection_projected_features.shape
    != (
        144,
        144,
    )
    or len(
        fit_writer_set
    ) != 145
    or len(
        selection_writer_set
    ) != 36
    or len(
        fit_writer_set
        & selection_writer_set
    ) != 0
    or not representation_alignment_audit[
        "raw_features_unit_normalized"
    ]
    or not representation_alignment_audit[
        "projected_features_unit_normalized"
    ]
    or representation_alignment_audit[
        "fit_script_counts"
    ]
    != {
        "Arabic": 290,
        "English": 290,
    }
    or representation_alignment_audit[
        "selection_script_counts"
    ]
    != {
        "Arabic": 72,
        "English": 72,
    }
    or representation_alignment_audit[
        "fit_text_condition_counts"
    ]
    != {
        "variable": 290,
        "same_fixed": 290,
    }
    or representation_alignment_audit[
        "selection_text_condition_counts"
    ]
    != {
        "variable": 72,
        "same_fixed": 72,
    }
):
    raise RuntimeError(
        "Notebook 28 representation alignment failed."
    )

{
  "source_artifact": "reports/dinov2s_student_baseline/dinov2s_final_refit_aligned_features.npz",
  "source_artifact_pages": 724,
  "source_artifact_writers": 181,
  "source_artifact_contains_only_fit_and_selection": true,
  "fit_pages": 580,
  "fit_writers": 145,
  "selection_pages": 144,
  "selection_writers": 36,
  "fit_selection_writer_overlap": 0,
  "raw_feature_dimension": 384,
  "projected_feature_dimension": 144,
  "fit_raw_shape": [
    580,
    384
  ],
  "selection_raw_shape": [
    144,
    384
  ],
  "fit_projected_shape": [
    580,
    144
  ],
  "selection_projected_shape": [
    144,
    144
  ],
  "fit_raw_norm_min": 0.9999998807907104,
  "fit_raw_norm_mean": 1.0,
  "fit_raw_norm_max": 1.0000001192092896,
  "selection_raw_norm_min": 0.9999998807907104,
  "selection_raw_norm_mean": 1.0,
  "selection_raw_norm_max": 1.0000001192092896,
  "fit_projected_norm_min": 0.9999998807907104,
  "fit_projected_norm_mean": 1.0,
  "fit_projected_norm_max": 1.0000001192092896,
  "se

In [4]:
PROBE_RANDOM_STATE = 42
PROBE_C = 1.0
PROBE_MAX_ITER = 5000
PROBE_SOLVER = "lbfgs"


def run_fixed_binary_linear_probe(
    fit_features,
    fit_labels,
    evaluation_features,
    evaluation_labels,
):
    pipeline = Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=PROBE_C,
                    penalty="l2",
                    solver=PROBE_SOLVER,
                    max_iter=PROBE_MAX_ITER,
                    random_state=PROBE_RANDOM_STATE,
                ),
            ),
        ]
    )

    pipeline.fit(
        fit_features,
        fit_labels,
    )

    classifier = (
        pipeline.named_steps[
            "classifier"
        ]
    )

    evaluation_probabilities = (
        pipeline.predict_proba(
            evaluation_features
        )[
            :,
            1,
        ]
    )

    evaluation_predictions = (
        pipeline.predict(
            evaluation_features
        )
    )

    fit_probabilities = (
        pipeline.predict_proba(
            fit_features
        )[
            :,
            1,
        ]
    )

    fit_predictions = (
        pipeline.predict(
            fit_features
        )
    )

    fit_auc = float(
        roc_auc_score(
            fit_labels,
            fit_probabilities,
        )
    )

    evaluation_auc = float(
        roc_auc_score(
            evaluation_labels,
            evaluation_probabilities,
        )
    )

    fit_accuracy = float(
        accuracy_score(
            fit_labels,
            fit_predictions,
        )
    )

    evaluation_accuracy = float(
        accuracy_score(
            evaluation_labels,
            evaluation_predictions,
        )
    )

    evaluation_balanced_accuracy = float(
        balanced_accuracy_score(
            evaluation_labels,
            evaluation_predictions,
        )
    )

    evaluation_confusion = (
        confusion_matrix(
            evaluation_labels,
            evaluation_predictions,
            labels=[
                0,
                1,
            ],
        )
    )

    coefficient_norm = float(
        np.linalg.norm(
            classifier.coef_
        )
    )

    iterations = int(
        np.max(
            classifier.n_iter_
        )
    )

    return {
        "pipeline": pipeline,
        "fit_auc": float(
            fit_auc
        ),
        "fit_accuracy": float(
            fit_accuracy
        ),
        "evaluation_auc": float(
            evaluation_auc
        ),
        "evaluation_accuracy": float(
            evaluation_accuracy
        ),
        "evaluation_balanced_accuracy": float(
            evaluation_balanced_accuracy
        ),
        "evaluation_confusion_matrix": (
            evaluation_confusion
            .astype(int)
            .tolist()
        ),
        "coefficient_norm": float(
            coefficient_norm
        ),
        "iterations": int(
            iterations
        ),
        "converged": bool(
            iterations
            < PROBE_MAX_ITER
        ),
        "evaluation_probabilities": (
            evaluation_probabilities
        ),
        "evaluation_predictions": (
            evaluation_predictions
        ),
    }


raw_script_probe = (
    run_fixed_binary_linear_probe(
        fit_features=fit_raw_features,
        fit_labels=fit_script_labels,
        evaluation_features=(
            selection_raw_features
        ),
        evaluation_labels=(
            selection_script_labels
        ),
    )
)


projected_script_probe = (
    run_fixed_binary_linear_probe(
        fit_features=(
            fit_projected_features
        ),
        fit_labels=fit_script_labels,
        evaluation_features=(
            selection_projected_features
        ),
        evaluation_labels=(
            selection_script_labels
        ),
    )
)


script_probe_comparison_df = pd.DataFrame(
    [
        {
            "representation": (
                "cached_normalized_dinov2s_384d"
            ),
            "dimension": 384,
            "fit_auc": float(
                raw_script_probe[
                    "fit_auc"
                ]
            ),
            "fit_accuracy": float(
                raw_script_probe[
                    "fit_accuracy"
                ]
            ),
            "selection_auc": float(
                raw_script_probe[
                    "evaluation_auc"
                ]
            ),
            "selection_accuracy": float(
                raw_script_probe[
                    "evaluation_accuracy"
                ]
            ),
            "selection_balanced_accuracy": float(
                raw_script_probe[
                    "evaluation_balanced_accuracy"
                ]
            ),
            "iterations": int(
                raw_script_probe[
                    "iterations"
                ]
            ),
            "converged": bool(
                raw_script_probe[
                    "converged"
                ]
            ),
        },
        {
            "representation": (
                "writer_projection_144d"
            ),
            "dimension": 144,
            "fit_auc": float(
                projected_script_probe[
                    "fit_auc"
                ]
            ),
            "fit_accuracy": float(
                projected_script_probe[
                    "fit_accuracy"
                ]
            ),
            "selection_auc": float(
                projected_script_probe[
                    "evaluation_auc"
                ]
            ),
            "selection_accuracy": float(
                projected_script_probe[
                    "evaluation_accuracy"
                ]
            ),
            "selection_balanced_accuracy": float(
                projected_script_probe[
                    "evaluation_balanced_accuracy"
                ]
            ),
            "iterations": int(
                projected_script_probe[
                    "iterations"
                ]
            ),
            "converged": bool(
                projected_script_probe[
                    "converged"
                ]
            ),
        },
    ]
)


raw_script_selection_auc = float(
    raw_script_probe[
        "evaluation_auc"
    ]
)

projected_script_selection_auc = float(
    projected_script_probe[
        "evaluation_auc"
    ]
)

script_auc_change_after_projection = float(
    projected_script_selection_auc
    - raw_script_selection_auc
)

raw_script_accuracy = float(
    raw_script_probe[
        "evaluation_accuracy"
    ]
)

projected_script_accuracy = float(
    projected_script_probe[
        "evaluation_accuracy"
    ]
)

script_accuracy_change_after_projection = float(
    projected_script_accuracy
    - raw_script_accuracy
)


script_probe_prediction_df = (
    selection_rows[
        [
            "filename",
            "writer",
            "page_id",
            "script_name",
            "script_label",
        ]
    ]
    .copy()
)

script_probe_prediction_df[
    "raw_probability_english"
] = (
    raw_script_probe[
        "evaluation_probabilities"
    ]
)

script_probe_prediction_df[
    "raw_prediction"
] = (
    raw_script_probe[
        "evaluation_predictions"
    ]
)

script_probe_prediction_df[
    "projected_probability_english"
] = (
    projected_script_probe[
        "evaluation_probabilities"
    ]
)

script_probe_prediction_df[
    "projected_prediction"
] = (
    projected_script_probe[
        "evaluation_predictions"
    ]
)


script_probe_summary = {
    "probe_target": (
        "script"
    ),
    "negative_class": (
        "Arabic"
    ),
    "positive_class": (
        "English"
    ),
    "probe_fit_writers": 145,
    "probe_fit_pages": 580,
    "probe_evaluation_writers": 36,
    "probe_evaluation_pages": 144,
    "writer_overlap": 0,
    "pipeline": (
        "StandardScaler fitted on fit only "
        "-> L2 logistic regression"
    ),
    "logistic_regression_c": float(
        PROBE_C
    ),
    "solver": (
        PROBE_SOLVER
    ),
    "maximum_iterations": int(
        PROBE_MAX_ITER
    ),
    "raw_dinov2s_dimension": 384,
    "raw_fit_auc": float(
        raw_script_probe[
            "fit_auc"
        ]
    ),
    "raw_fit_accuracy": float(
        raw_script_probe[
            "fit_accuracy"
        ]
    ),
    "raw_selection_auc": float(
        raw_script_selection_auc
    ),
    "raw_selection_accuracy": float(
        raw_script_accuracy
    ),
    "raw_selection_balanced_accuracy": float(
        raw_script_probe[
            "evaluation_balanced_accuracy"
        ]
    ),
    "raw_selection_confusion_matrix": (
        raw_script_probe[
            "evaluation_confusion_matrix"
        ]
    ),
    "raw_probe_converged": bool(
        raw_script_probe[
            "converged"
        ]
    ),
    "projected_dimension": 144,
    "projected_fit_auc": float(
        projected_script_probe[
            "fit_auc"
        ]
    ),
    "projected_fit_accuracy": float(
        projected_script_probe[
            "fit_accuracy"
        ]
    ),
    "projected_selection_auc": float(
        projected_script_selection_auc
    ),
    "projected_selection_accuracy": float(
        projected_script_accuracy
    ),
    "projected_selection_balanced_accuracy": float(
        projected_script_probe[
            "evaluation_balanced_accuracy"
        ]
    ),
    "projected_selection_confusion_matrix": (
        projected_script_probe[
            "evaluation_confusion_matrix"
        ]
    ),
    "projected_probe_converged": bool(
        projected_script_probe[
            "converged"
        ]
    ),
    "projected_minus_raw_selection_auc": float(
        script_auc_change_after_projection
    ),
    "projected_minus_raw_selection_accuracy": float(
        script_accuracy_change_after_projection
    ),
    "probe_protocol_fixed_before_evaluation": True,
    "probe_hyperparameter_tuning_allowed": False,
    "writer_verification_model_modified": False,
    "monitor_features_used": False,
    "monitor_evaluated": False,
    "validation_used": False,
    "official_test_used": False,
}


script_probe_comparison_df.to_csv(
    REPORT_DIR
    / "dinov2s_script_probe_comparison.csv",
    index=False,
)

script_probe_prediction_df.to_csv(
    REPORT_DIR
    / "dinov2s_script_probe_selection_predictions.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_script_probe_summary.json",
    "w",
) as file:
    json.dump(
        script_probe_summary,
        file,
        indent=2,
    )


print(
    json.dumps(
        script_probe_summary,
        indent=2,
    )
)

print(
    "\nScript probe comparison:"
)

print(
    script_probe_comparison_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    not raw_script_probe[
        "converged"
    ]
    or not projected_script_probe[
        "converged"
    ]
    or not np.isfinite(
        [
            raw_script_selection_auc,
            projected_script_selection_auc,
            raw_script_accuracy,
            projected_script_accuracy,
        ]
    ).all()
    or raw_script_probe[
        "evaluation_confusion_matrix"
    ]
    is None
    or projected_script_probe[
        "evaluation_confusion_matrix"
    ]
    is None
):
    raise RuntimeError(
        "Notebook 28 script probe audit failed."
    )

{
  "probe_target": "script",
  "negative_class": "Arabic",
  "positive_class": "English",
  "probe_fit_writers": 145,
  "probe_fit_pages": 580,
  "probe_evaluation_writers": 36,
  "probe_evaluation_pages": 144,
  "writer_overlap": 0,
  "pipeline": "StandardScaler fitted on fit only -> L2 logistic regression",
  "logistic_regression_c": 1.0,
  "solver": "lbfgs",
  "maximum_iterations": 5000,
  "raw_dinov2s_dimension": 384,
  "raw_fit_auc": 1.0,
  "raw_fit_accuracy": 1.0,
  "raw_selection_auc": 1.0,
  "raw_selection_accuracy": 0.9861111111111112,
  "raw_selection_balanced_accuracy": 0.9861111111111112,
  "raw_selection_confusion_matrix": [
    [
      72,
      0
    ],
    [
      2,
      70
    ]
  ],
  "raw_probe_converged": true,
  "projected_dimension": 144,
  "projected_fit_auc": 0.9053269916765755,
  "projected_fit_accuracy": 0.8258620689655173,
  "projected_selection_auc": 0.757908950617284,
  "projected_selection_accuracy": 0.6944444444444444,
  "projected_selection_balanced_a

/home/arijit/Documents/handwriting-cross-script-research/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/arijit/Documents/handwriting-cross-script-research/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf inst

In [5]:
raw_text_condition_probe = (
    run_fixed_binary_linear_probe(
        fit_features=fit_raw_features,
        fit_labels=fit_text_condition_labels,
        evaluation_features=(
            selection_raw_features
        ),
        evaluation_labels=(
            selection_text_condition_labels
        ),
    )
)


projected_text_condition_probe = (
    run_fixed_binary_linear_probe(
        fit_features=(
            fit_projected_features
        ),
        fit_labels=fit_text_condition_labels,
        evaluation_features=(
            selection_projected_features
        ),
        evaluation_labels=(
            selection_text_condition_labels
        ),
    )
)


text_condition_probe_comparison_df = pd.DataFrame(
    [
        {
            "representation": (
                "cached_normalized_dinov2s_384d"
            ),
            "dimension": 384,
            "fit_auc": float(
                raw_text_condition_probe[
                    "fit_auc"
                ]
            ),
            "fit_accuracy": float(
                raw_text_condition_probe[
                    "fit_accuracy"
                ]
            ),
            "selection_auc": float(
                raw_text_condition_probe[
                    "evaluation_auc"
                ]
            ),
            "selection_accuracy": float(
                raw_text_condition_probe[
                    "evaluation_accuracy"
                ]
            ),
            "selection_balanced_accuracy": float(
                raw_text_condition_probe[
                    "evaluation_balanced_accuracy"
                ]
            ),
            "iterations": int(
                raw_text_condition_probe[
                    "iterations"
                ]
            ),
            "converged": bool(
                raw_text_condition_probe[
                    "converged"
                ]
            ),
        },
        {
            "representation": (
                "writer_projection_144d"
            ),
            "dimension": 144,
            "fit_auc": float(
                projected_text_condition_probe[
                    "fit_auc"
                ]
            ),
            "fit_accuracy": float(
                projected_text_condition_probe[
                    "fit_accuracy"
                ]
            ),
            "selection_auc": float(
                projected_text_condition_probe[
                    "evaluation_auc"
                ]
            ),
            "selection_accuracy": float(
                projected_text_condition_probe[
                    "evaluation_accuracy"
                ]
            ),
            "selection_balanced_accuracy": float(
                projected_text_condition_probe[
                    "evaluation_balanced_accuracy"
                ]
            ),
            "iterations": int(
                projected_text_condition_probe[
                    "iterations"
                ]
            ),
            "converged": bool(
                projected_text_condition_probe[
                    "converged"
                ]
            ),
        },
    ]
)


raw_text_selection_auc = float(
    raw_text_condition_probe[
        "evaluation_auc"
    ]
)

projected_text_selection_auc = float(
    projected_text_condition_probe[
        "evaluation_auc"
    ]
)

text_auc_change_after_projection = float(
    projected_text_selection_auc
    - raw_text_selection_auc
)

raw_text_accuracy = float(
    raw_text_condition_probe[
        "evaluation_accuracy"
    ]
)

projected_text_accuracy = float(
    projected_text_condition_probe[
        "evaluation_accuracy"
    ]
)

text_accuracy_change_after_projection = float(
    projected_text_accuracy
    - raw_text_accuracy
)


text_condition_prediction_df = (
    selection_rows[
        [
            "filename",
            "writer",
            "page_id",
            "text_condition_name",
            "text_condition_label",
        ]
    ]
    .copy()
)

text_condition_prediction_df[
    "raw_probability_same_fixed"
] = (
    raw_text_condition_probe[
        "evaluation_probabilities"
    ]
)

text_condition_prediction_df[
    "raw_prediction"
] = (
    raw_text_condition_probe[
        "evaluation_predictions"
    ]
)

text_condition_prediction_df[
    "projected_probability_same_fixed"
] = (
    projected_text_condition_probe[
        "evaluation_probabilities"
    ]
)

text_condition_prediction_df[
    "projected_prediction"
] = (
    projected_text_condition_probe[
        "evaluation_predictions"
    ]
)


text_condition_probe_summary = {
    "probe_target": (
        "text_condition"
    ),
    "negative_class": (
        "variable"
    ),
    "positive_class": (
        "same_fixed"
    ),
    "probe_fit_writers": 145,
    "probe_fit_pages": 580,
    "probe_evaluation_writers": 36,
    "probe_evaluation_pages": 144,
    "writer_overlap": 0,
    "pipeline": (
        "StandardScaler fitted on fit only "
        "-> L2 logistic regression"
    ),
    "logistic_regression_c": float(
        PROBE_C
    ),
    "solver": (
        PROBE_SOLVER
    ),
    "maximum_iterations": int(
        PROBE_MAX_ITER
    ),
    "raw_dinov2s_dimension": 384,
    "raw_fit_auc": float(
        raw_text_condition_probe[
            "fit_auc"
        ]
    ),
    "raw_fit_accuracy": float(
        raw_text_condition_probe[
            "fit_accuracy"
        ]
    ),
    "raw_selection_auc": float(
        raw_text_selection_auc
    ),
    "raw_selection_accuracy": float(
        raw_text_accuracy
    ),
    "raw_selection_balanced_accuracy": float(
        raw_text_condition_probe[
            "evaluation_balanced_accuracy"
        ]
    ),
    "raw_selection_confusion_matrix": (
        raw_text_condition_probe[
            "evaluation_confusion_matrix"
        ]
    ),
    "raw_probe_converged": bool(
        raw_text_condition_probe[
            "converged"
        ]
    ),
    "projected_dimension": 144,
    "projected_fit_auc": float(
        projected_text_condition_probe[
            "fit_auc"
        ]
    ),
    "projected_fit_accuracy": float(
        projected_text_condition_probe[
            "fit_accuracy"
        ]
    ),
    "projected_selection_auc": float(
        projected_text_selection_auc
    ),
    "projected_selection_accuracy": float(
        projected_text_accuracy
    ),
    "projected_selection_balanced_accuracy": float(
        projected_text_condition_probe[
            "evaluation_balanced_accuracy"
        ]
    ),
    "projected_selection_confusion_matrix": (
        projected_text_condition_probe[
            "evaluation_confusion_matrix"
        ]
    ),
    "projected_probe_converged": bool(
        projected_text_condition_probe[
            "converged"
        ]
    ),
    "projected_minus_raw_selection_auc": float(
        text_auc_change_after_projection
    ),
    "projected_minus_raw_selection_accuracy": float(
        text_accuracy_change_after_projection
    ),
    "same_probe_protocol_as_script_audit": True,
    "probe_protocol_fixed_before_evaluation": True,
    "probe_hyperparameter_tuning_allowed": False,
    "writer_verification_model_modified": False,
    "monitor_features_used": False,
    "monitor_evaluated": False,
    "validation_used": False,
    "official_test_used": False,
}


text_condition_probe_comparison_df.to_csv(
    REPORT_DIR
    / "dinov2s_text_condition_probe_comparison.csv",
    index=False,
)

text_condition_prediction_df.to_csv(
    REPORT_DIR
    / "dinov2s_text_condition_probe_selection_predictions.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_text_condition_probe_summary.json",
    "w",
) as file:
    json.dump(
        text_condition_probe_summary,
        file,
        indent=2,
    )


print(
    json.dumps(
        text_condition_probe_summary,
        indent=2,
    )
)

print(
    "\nText-condition probe comparison:"
)

print(
    text_condition_probe_comparison_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    not raw_text_condition_probe[
        "converged"
    ]
    or not projected_text_condition_probe[
        "converged"
    ]
    or not np.isfinite(
        [
            raw_text_selection_auc,
            projected_text_selection_auc,
            raw_text_accuracy,
            projected_text_accuracy,
        ]
    ).all()
):
    raise RuntimeError(
        "Notebook 28 text-condition probe audit failed."
    )

{
  "probe_target": "text_condition",
  "negative_class": "variable",
  "positive_class": "same_fixed",
  "probe_fit_writers": 145,
  "probe_fit_pages": 580,
  "probe_evaluation_writers": 36,
  "probe_evaluation_pages": 144,
  "writer_overlap": 0,
  "pipeline": "StandardScaler fitted on fit only -> L2 logistic regression",
  "logistic_regression_c": 1.0,
  "solver": "lbfgs",
  "maximum_iterations": 5000,
  "raw_dinov2s_dimension": 384,
  "raw_fit_auc": 1.0,
  "raw_fit_accuracy": 1.0,
  "raw_selection_auc": 0.9972993827160495,
  "raw_selection_accuracy": 0.9861111111111112,
  "raw_selection_balanced_accuracy": 0.9861111111111112,
  "raw_selection_confusion_matrix": [
    [
      72,
      0
    ],
    [
      2,
      70
    ]
  ],
  "raw_probe_converged": true,
  "projected_dimension": 144,
  "projected_fit_auc": 0.9016171224732461,
  "projected_fit_accuracy": 0.8258620689655173,
  "projected_selection_auc": 0.7924382716049383,
  "projected_selection_accuracy": 0.7013888888888888,
  "p

/home/arijit/Documents/handwriting-cross-script-research/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/arijit/Documents/handwriting-cross-script-research/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf inst

In [6]:
nuisance_comparison_df = pd.DataFrame(
    [
        {
            "nuisance": "script",
            "raw_selection_auc": float(
                raw_script_probe[
                    "evaluation_auc"
                ]
            ),
            "projected_selection_auc": float(
                projected_script_probe[
                    "evaluation_auc"
                ]
            ),
            "auc_change_after_projection": float(
                projected_script_probe[
                    "evaluation_auc"
                ]
                - raw_script_probe[
                    "evaluation_auc"
                ]
            ),
            "raw_selection_accuracy": float(
                raw_script_probe[
                    "evaluation_accuracy"
                ]
            ),
            "projected_selection_accuracy": float(
                projected_script_probe[
                    "evaluation_accuracy"
                ]
            ),
            "accuracy_change_after_projection": float(
                projected_script_probe[
                    "evaluation_accuracy"
                ]
                - raw_script_probe[
                    "evaluation_accuracy"
                ]
            ),
        },
        {
            "nuisance": "text_condition",
            "raw_selection_auc": float(
                raw_text_condition_probe[
                    "evaluation_auc"
                ]
            ),
            "projected_selection_auc": float(
                projected_text_condition_probe[
                    "evaluation_auc"
                ]
            ),
            "auc_change_after_projection": float(
                projected_text_condition_probe[
                    "evaluation_auc"
                ]
                - raw_text_condition_probe[
                    "evaluation_auc"
                ]
            ),
            "raw_selection_accuracy": float(
                raw_text_condition_probe[
                    "evaluation_accuracy"
                ]
            ),
            "projected_selection_accuracy": float(
                projected_text_condition_probe[
                    "evaluation_accuracy"
                ]
            ),
            "accuracy_change_after_projection": float(
                projected_text_condition_probe[
                    "evaluation_accuracy"
                ]
                - raw_text_condition_probe[
                    "evaluation_accuracy"
                ]
            ),
        },
    ]
)


raw_mean_nuisance_auc = float(
    nuisance_comparison_df[
        "raw_selection_auc"
    ].mean()
)

projected_mean_nuisance_auc = float(
    nuisance_comparison_df[
        "projected_selection_auc"
    ].mean()
)

mean_nuisance_auc_change = float(
    nuisance_comparison_df[
        "auc_change_after_projection"
    ].mean()
)

all_nuisances_reduced = bool(
    (
        nuisance_comparison_df[
            "auc_change_after_projection"
        ]
        < 0.0
    ).all()
)

all_projected_nuisance_auc_above_chance = bool(
    (
        nuisance_comparison_df[
            "projected_selection_auc"
        ]
        > 0.5
    ).all()
)


dinov2s_nuisance_accessibility_verdict = {
    "representation_comparison": (
        "cached normalized DINOv2-S 384-D "
        "vs writer projection 144-D"
    ),
    "probe_fit_writers": 145,
    "probe_evaluation_writers": 36,
    "writer_overlap": 0,
    "raw_script_selection_auc": float(
        raw_script_probe[
            "evaluation_auc"
        ]
    ),
    "projected_script_selection_auc": float(
        projected_script_probe[
            "evaluation_auc"
        ]
    ),
    "script_auc_change_after_projection": float(
        script_auc_change_after_projection
    ),
    "raw_text_condition_selection_auc": float(
        raw_text_condition_probe[
            "evaluation_auc"
        ]
    ),
    "projected_text_condition_selection_auc": float(
        projected_text_condition_probe[
            "evaluation_auc"
        ]
    ),
    "text_condition_auc_change_after_projection": float(
        text_auc_change_after_projection
    ),
    "raw_mean_nuisance_auc": float(
        raw_mean_nuisance_auc
    ),
    "projected_mean_nuisance_auc": float(
        projected_mean_nuisance_auc
    ),
    "mean_nuisance_auc_change_after_projection": float(
        mean_nuisance_auc_change
    ),
    "all_measured_nuisances_reduced_after_projection": bool(
        all_nuisances_reduced
    ),
    "all_projected_nuisance_aucs_above_chance": bool(
        all_projected_nuisance_auc_above_chance
    ),
    "writer_projection_substantially_reduces_linear_nuisance_accessibility": bool(
        all_nuisances_reduced
        and mean_nuisance_auc_change
        < -0.15
    ),
    "writer_projection_is_fully_nuisance_invariant": False,
    "interpretation": (
        "the writer-trained projection substantially reduces "
        "linearly accessible script and text-condition information, "
        "but residual nuisance information remains accessible"
    ),
    "causal_dependence_claim_supported": False,
    "reason_causal_dependence_not_claimed": (
        "linear-probe accessibility does not establish that "
        "the writer verifier causally relies on the nuisance variable"
    ),
    "explicit_nuisance_suppression_scientifically_testable_next": True,
    "recommended_next_experiment": (
        "controlled nuisance-suppression experiment on the "
        "DINOv2-S projection student, with the current projection "
        "baseline retained as the frozen reference"
    ),
    "writer_verification_model_modified_in_this_notebook": False,
    "probe_hyperparameter_tuning_performed": False,
    "monitor_features_used": False,
    "monitor_evaluated": False,
    "validation_used": False,
    "official_test_used": False,
    "nuisance_audit_complete": True,
}


nuisance_comparison_df.to_csv(
    REPORT_DIR
    / "dinov2s_nuisance_accessibility_comparison.csv",
    index=False,
)

with open(
    REPORT_DIR
    / "dinov2s_nuisance_accessibility_verdict.json",
    "w",
) as file:
    json.dump(
        dinov2s_nuisance_accessibility_verdict,
        file,
        indent=2,
    )


print(
    json.dumps(
        dinov2s_nuisance_accessibility_verdict,
        indent=2,
    )
)

print(
    "\nNuisance accessibility comparison:"
)

print(
    nuisance_comparison_df
    .round(
        6
    )
    .to_string(
        index=False
    )
)


if (
    not np.isfinite(
        nuisance_comparison_df[
            [
                "raw_selection_auc",
                "projected_selection_auc",
                "auc_change_after_projection",
            ]
        ].to_numpy()
    ).all()
    or not all_nuisances_reduced
):
    raise RuntimeError(
        "Notebook 28 nuisance-accessibility verdict audit failed."
    )

{
  "representation_comparison": "cached normalized DINOv2-S 384-D vs writer projection 144-D",
  "probe_fit_writers": 145,
  "probe_evaluation_writers": 36,
  "writer_overlap": 0,
  "raw_script_selection_auc": 1.0,
  "projected_script_selection_auc": 0.757908950617284,
  "script_auc_change_after_projection": -0.24209104938271597,
  "raw_text_condition_selection_auc": 0.9972993827160495,
  "projected_text_condition_selection_auc": 0.7924382716049383,
  "text_condition_auc_change_after_projection": -0.20486111111111116,
  "raw_mean_nuisance_auc": 0.9986496913580247,
  "projected_mean_nuisance_auc": 0.7751736111111112,
  "mean_nuisance_auc_change_after_projection": -0.22347608024691357,
  "all_measured_nuisances_reduced_after_projection": true,
  "all_projected_nuisance_aucs_above_chance": true,
  "writer_projection_substantially_reduces_linear_nuisance_accessibility": true,
  "writer_projection_is_fully_nuisance_invariant": false,
  "interpretation": "the writer-trained projection subst

## Final Summary

This notebook examined whether script and text-condition information remained linearly accessible in the strong DINOv2-S writer representation established in Notebook 27.

The audit compared two representations under an identical writer-disjoint linear-probe protocol:

1. **cached normalized DINOv2-S 384-D features**
2. **writer-trained DINOv2-S projection 144-D features**

The projection checkpoint came from the development-stage seed-42 model trained only on the **145 fit writers**. The final 181-writer refit checkpoint was deliberately excluded because it had already seen the 36 selection writers.

The nuisance probes were fitted on:

**145 fit writers / 580 pages**

and evaluated on:

**36 unseen selection writers / 144 pages**

with zero writer overlap.

No monitor, validation, or official-test data were used.

### Script Accessibility

For Arabic-versus-English script classification, the cached DINOv2-S representation produced:

- Fit ROC-AUC: **1.00000**
- Selection ROC-AUC: **1.00000**
- Selection accuracy: **0.98611**

Script identity was therefore almost perfectly linearly accessible in the original frozen DINOv2-S feature space.

After the writer-trained 384→144 projection, the corresponding result became:

- Fit ROC-AUC: **0.90533**
- Selection ROC-AUC: **0.75791**
- Selection accuracy: **0.69444**

The writer projection reduced writer-disjoint script ROC-AUC by:

**−0.24209**

and reduced classification accuracy by approximately:

**−0.29167**

This is a substantial reduction in linearly accessible script information.

### Text-Condition Accessibility

For variable-text versus same/fixed-text classification, the cached DINOv2-S representation produced:

- Fit ROC-AUC: **1.00000**
- Selection ROC-AUC: **0.99730**
- Selection accuracy: **0.98611**

Text condition was therefore also almost perfectly linearly accessible in the original representation.

After the writer-trained projection, performance became:

- Fit ROC-AUC: **0.90162**
- Selection ROC-AUC: **0.79244**
- Selection accuracy: **0.70139**

The projection reduced writer-disjoint text-condition ROC-AUC by:

**−0.20486**

and reduced classification accuracy by approximately:

**−0.28472**

### Combined Nuisance Accessibility

Across the two measured nuisance variables, the mean writer-disjoint probe ROC-AUC changed from:

**0.99865 → 0.77517**

corresponding to a mean reduction of:

**−0.22348 ROC-AUC**

Both measured nuisance variables became substantially less linearly accessible after writer-metric projection.

However, both projected probe AUCs remained clearly above chance:

- Script: **0.75791**
- Text condition: **0.79244**

The projected writer representation should therefore not be described as fully script-invariant or nuisance-invariant.

### Scientific Interpretation

The result provides an important complement to Notebook 27.

The original DINOv2-S representation contains extremely strong script and text-condition structure. However, the writer-verification projection learned without any explicit nuisance-adversarial objective already removes a substantial amount of this linearly accessible nuisance structure.

This suggests that the writer-centric metric objective itself encourages partial nuisance suppression while retaining the writer information required for cross-script verification.

At the same time, residual nuisance information remains sufficiently accessible that an explicit nuisance-suppression experiment is scientifically justified.

The result does **not** establish that the writer verifier causally relies on the remaining script or text-condition information.

Linear probes measure whether information can be decoded by a simple classifier; they do not establish how that information contributes to verification decisions.

Therefore, the correct conclusion is:

> The DINOv2-S writer projection substantially reduces, but does not eliminate, linearly accessible nuisance information.

### Next Research Question

The next experiment should test whether explicit nuisance suppression can reduce the remaining script accessibility without damaging the strong writer-verification performance established by the DINOv2-S projection baseline.

To keep the experiment controlled, script should be treated as the primary adversarial nuisance because cross-script invariance is the central research objective.

Text-condition accessibility should remain a secondary diagnostic rather than introducing a second adversarial objective at the same time.

The appropriate next experiment is therefore:

**frozen DINOv2-S 384-D representation → trainable 144-D writer projection + controlled script-adversarial suppression**

with the Notebook 27 projection student retained unchanged as the reference baseline.

### Protocol Status

- Probe fit writers: **145**
- Probe evaluation writers: **36**
- Writer overlap: **0**
- Raw DINOv2-S script accessibility measured: **Yes**
- Projected script accessibility measured: **Yes**
- Raw text-condition accessibility measured: **Yes**
- Projected text-condition accessibility measured: **Yes**
- Probe hyperparameter tuning performed: **No**
- Writer-verification model modified: **No**
- Monitor used: **No**
- Validation used: **No**
- Official test used: **No**
- Nuisance accessibility audit complete: **Yes**

Notebook 28 therefore closes with evidence of **substantial but incomplete nuisance suppression** in the DINOv2-S writer projection.